In [1]:
import random

In [2]:
nums = [2**i for i in range(16)]
nums[0] = 0

combs = [[a, b, c, d] for a in nums for b in nums for c in nums for d in nums]
len(combs)

65536

In [4]:
num_to_4bit = {
    2**i if i >= 1 else 0: format(i, 'b').zfill(4) for i in range(0, 16)
}

fourbit_to_num = {
    format(i, 'b').zfill(4): 2**i if i >= 1 else 0 for i in range(0, 16)
}

def encode_list_to_str(l):
    s = "".join(num_to_4bit[n] for n in l)
    return s

def decode_str_to_list(s):
    s = s.zfill(16)
    a, b, c, d = [s[i*4:(i+1)*4] for i in range(4)]
    return [fourbit_to_num[a], fourbit_to_num[b], fourbit_to_num[c], fourbit_to_num[d]]

num_to_4bit

{0: '0000',
 2: '0001',
 4: '0010',
 8: '0011',
 16: '0100',
 32: '0101',
 64: '0110',
 128: '0111',
 256: '1000',
 512: '1001',
 1024: '1010',
 2048: '1011',
 4096: '1100',
 8192: '1101',
 16384: '1110',
 32768: '1111'}

In [5]:
def shift_left(l):
    new = [0, 0, 0, 0]
    new_i = 0
    for num in l:
        if num != 0:
            new[new_i] = num
            new_i += 1
    return new

def merge(l):
    score = 0
    for i in range(len(l) - 1):
        if l[i] == l[i+1] and l[i] != 32768:
            score += l[i]
            l[i] = l[i] * 2
            l[i+1] = 0
    return l, score


def do_left(l):
    temp = shift_left(l)
    merged, score = merge(temp)
    return shift_left(merged), score


do_left([2, 4, 4, 2])

([2, 8, 2, 0], 4)

In [6]:
unencoded_lut_row = {
    ",".join(str(n) for n in l): do_left(l)[0] for l in combs
}

unencoded_lut_score = {
    ",".join(str(n) for n in l): do_left(l)[1] for l in combs
}

In [7]:
unencoded_lut_score

{'0,0,0,0': 0,
 '0,0,0,2': 0,
 '0,0,0,4': 0,
 '0,0,0,8': 0,
 '0,0,0,16': 0,
 '0,0,0,32': 0,
 '0,0,0,64': 0,
 '0,0,0,128': 0,
 '0,0,0,256': 0,
 '0,0,0,512': 0,
 '0,0,0,1024': 0,
 '0,0,0,2048': 0,
 '0,0,0,4096': 0,
 '0,0,0,8192': 0,
 '0,0,0,16384': 0,
 '0,0,0,32768': 0,
 '0,0,2,0': 0,
 '0,0,2,2': 2,
 '0,0,2,4': 0,
 '0,0,2,8': 0,
 '0,0,2,16': 0,
 '0,0,2,32': 0,
 '0,0,2,64': 0,
 '0,0,2,128': 0,
 '0,0,2,256': 0,
 '0,0,2,512': 0,
 '0,0,2,1024': 0,
 '0,0,2,2048': 0,
 '0,0,2,4096': 0,
 '0,0,2,8192': 0,
 '0,0,2,16384': 0,
 '0,0,2,32768': 0,
 '0,0,4,0': 0,
 '0,0,4,2': 0,
 '0,0,4,4': 4,
 '0,0,4,8': 0,
 '0,0,4,16': 0,
 '0,0,4,32': 0,
 '0,0,4,64': 0,
 '0,0,4,128': 0,
 '0,0,4,256': 0,
 '0,0,4,512': 0,
 '0,0,4,1024': 0,
 '0,0,4,2048': 0,
 '0,0,4,4096': 0,
 '0,0,4,8192': 0,
 '0,0,4,16384': 0,
 '0,0,4,32768': 0,
 '0,0,8,0': 0,
 '0,0,8,2': 0,
 '0,0,8,4': 0,
 '0,0,8,8': 8,
 '0,0,8,16': 0,
 '0,0,8,32': 0,
 '0,0,8,64': 0,
 '0,0,8,128': 0,
 '0,0,8,256': 0,
 '0,0,8,512': 0,
 '0,0,8,1024': 0,
 '0,0,8,2048': 0

In [218]:
encoded_lut_row_left = {
    int(encode_list_to_str(l), base = 2): int(encode_list_to_str(do_left(l)[0]), base = 2) for l in combs
}

encoded_lut_score_left = {
    int(encode_list_to_str(l), base = 2): do_left(l)[1] for l in combs
}

encoded_lut_row_right = {
    int(encode_list_to_str(l), base = 2): reverse_lut[encoded_lut_row[reverse_lut[int(encode_list_to_str(l), base = 2)]]] for l in combs
}

encoded_lut_score_right = {
    int(encode_list_to_str(l), base = 2): do_left(decode_str_to_list(bin(reverse_lut[int(encode_list_to_str(l), base = 2)])[2:]))[1] for l in combs
}

# ... inside generate_lut.py ...

print("Generating Row Info Table...")
row_info_table = {} 
# We will store a tuple or bitmask: (has_2048, can_move)
# But for speed, let's just store simple flags.
# 2 = Has 2048 (Win)
# 1 = Can Move (Not Over)
# 0 = Dead Row (Full and no merges)

for i in range(65536):
    # Decode to list [a, b, c, d]
    row = [
        (i >> 12) & 0xF, 
        (i >> 8) & 0xF, 
        (i >> 4) & 0xF, 
        i & 0xF
    ]
    
    has_2048 = False
    can_move = False
    
    # Check 1: 2048 (Value 11)
    if 11 in row:
        has_2048 = True
        
    # Check 2: Empty Space (0)
    if 0 in row:
        can_move = True
    else:
        # Check 3: Merges (only if no empty space found yet)
        # adjacent checks: 0-1, 1-2, 2-3
        if row[0] == row[1] or row[1] == row[2] or row[2] == row[3]:
            can_move = True
            
    # Encode result
    # We prioritize information. 
    # If it has 2048, that's the most important info.
    # But wait, your original code checks Win FIRST.
    
    info = {
        'win': has_2048,
        'move': can_move
    }
    row_info_table[i] = info

# Add 'info': row_info_table to your pickle data dictionary

luts = {'right': encoded_lut_row_right, 
        'left': encoded_lut_row_left, 
        'left_score': encoded_lut_score_left, 
        'right_score': encoded_lut_score_right,
        'row_info_table': row_info_table}
import pickle
with open("2048_lut.pkl", "wb") as f:
    pickle.dump(luts, f)

Generating Row Info Table...


In [221]:
row_info_table[int(encode_list_to_str([0, 0, 0, 0]), base = 2)]

{'win': False, 'move': True}

In [ ]:

encoded_lut_score = {
    int(encode_list_to_str(l), base = 2): reverse_lut[encoded_lut_row[reverse_lut[int(encode_list_to_str(l), base = 2)]]] for l in combs
}

In [64]:
encoded_lut_score[int(encode_list_to_str([2, 0, 2, 0]), base = 2)]

2

In [70]:
try:
    with open("2048_lut.pkl", "rb") as f:
        data = pickle.load(f)
        MOVES_LEFT = data["left"]
        MOVES_RIGHT = data["right"]
        LEFT_SCORES = data["left_score"]
        RIGHT_SCORES = data["right_score"]
        print("luts loaded successfully.")
except FileNotFoundError:
    print("Error: '2048_lut.pkl' not found. Please run generate_lut.py first.")
    # Optional: You could call the generation function here as a fallback
    exit()

luts loaded successfully.


In [10]:
def print_2048(rows):
    # 1. Calculate the max width needed for the numbers
    max_w = 0
    for row in rows:
        for e in row:
            if len(str(e)) > max_w:
                max_w = len(str(e))
    
    # 2. Calculate the total width of the box to make borders match
    # (cols * (number_width + 1 for padding)) + (cols - 1 for spaces between)
    cols = len(rows[0])
    line_width = (cols * (max_w + 1)) + (cols - 1)

    # 3. Print the Box
    print(' ' + '_' * line_width + ' ')  # Dynamic Top Border
    for row in rows:
        print(f'|{" ".join(f"{str(e):>{max_w + 1}}" for e in row)}|')
    print(' ' + '‾' * line_width + ' ')  # Dynamic Bottom Border

In [11]:
bin(((0b11111111 << 16) | 0b11111111) << 32), bin((0b11111111 << 24) | (0b11111111 << 8))

('0b11111111000000001111111100000000000000000000000000000000',
 '0b11111111000000001111111100000000')

In [48]:
def transpose(board):
    # first, swap the top right 2 * 2 and bottom left 2 * 2
    # num = a1 a2 a3 a4 b1 b2 b3 b4 c1 c2 c3 c4 d1 d2 d3 d4
    # swap a3 a4 b3 b4 and c1 c2 d1 d2

    top_right_mask = ((0b11111111 << 16) | 0b11111111) << 32
    bottom_left_mask = (0b11111111 << 24) | (0b11111111 << 8)

    top_right = (board & top_right_mask) >> 32
    bottom_left = (board & bottom_left_mask) >> 8

    diff = top_right ^ bottom_left

    mask = diff << 32 | diff << 8

    board ^= mask
    
    # now, tranpose the individual 2*2
    even_mask = (0b0000111100001111 << 48) | (0b0000111100001111 << 16)
    odd_mask = (0b1111000011110000 << 32) | 0b1111000011110000

    evens = (board & even_mask) >> 16
    odds = (board & odd_mask) >> 4

    diff = evens ^ odds

    mask = diff << 4 | diff << 16

    board ^= mask

    return board

In [85]:
CHOICES = list(encoded_lut_row.keys())

def make_test_board():
    n = 0
    for _ in range(4):
        n <<= 16
        row = random.choice(CHOICES)
        # print(row)
        n |= row
    return n

def int_to_board(num):
    rows = get_rows(num)
    rows = [decode_str_to_list(format(row, 'b')) for row in rows] # 2d array format
    print_2048(rows)

def get_rows(num):
    mask = 0b1111111111111111
    d = num & mask
    c = (num >> 16) & mask
    b = (num >> 32) & mask
    a = (num >> 48) & mask
    return a, b, c, d

# def reverse_row(num):
#     mask = 0b1111
#     res = 0
#     nums = [num & mask, (num >> 4) & mask, (num >> 8) & mask, (num >> 12) & mask]
#     res |= nums[0]
#     res <<= 4
#     res |= nums[1]
#     res <<= 4
#     res |= nums[2]
#     res <<= 4
#     res |= nums[3]
#     return res

# move left -> break into 16 bit chunks look up table
def move_left(num):
    # need to handle scoring too
    a, b, c, d = get_rows(num)
    res = 0
    res |= MOVES_LEFT[a]
    res <<= 16
    res |= MOVES_LEFT[b]
    res <<= 16
    res |= MOVES_LEFT[c]
    res <<= 16
    res |= MOVES_LEFT[d]

    score = LEFT_SCORES[a] + LEFT_SCORES[b] + LEFT_SCORES[c] + LEFT_SCORES[d]
    return res, score


# move right -> reverse, then look up table
# given a int thats 64 bits -> break into 16 bit chunks, reverse. look up each, then combine it back
def move_right(num):
    a, b, c, d = get_rows(num)

    res = 0
    res |= MOVES_RIGHT[a]
    res <<= 16
    res |= MOVES_RIGHT[b]
    res <<= 16
    res |= MOVES_RIGHT[c]
    res <<= 16
    res |= MOVES_RIGHT[d]

    score = RIGHT_SCORES[a] + RIGHT_SCORES[b] + RIGHT_SCORES[c] + RIGHT_SCORES[d]
    return res, score


def move_up(num):
    num = transpose(num)
    num, score = move_left(num)
    num = transpose(num)
    return num, score

def move_down(num):
    num = transpose(num)
    num, score = move_right(num)
    num = transpose(num)
    return num, score


test_board = make_test_board()
int_to_board(test_board)
test_board, score = move_right(test_board)
int_to_board(test_board)
print(score)

 ___________________________ 
|     0    256   2048      2|
|  8192      0   8192  16384|
|     2      2   1024    512|
|     4   8192    256  32768|
 ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾ 
 ___________________________ 
|     0    256   2048      2|
|     0      0  16384  16384|
|     0      4   1024    512|
|     4   8192    256  32768|
 ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾ 
8194


In [208]:
def new_board():
    pos = random.choices([i for i in range(16)], k=2)
    while pos[0] == pos[1]:
        pos = random.choices([i for i in range(16)], k = 2)
    return 0b01 << pos[0] * 4 | 0b01 << pos[1] * 4

int_to_board(new_board())

 ___________ 
| 0  0  2  2|
| 0  0  0  0|
| 0  0  0  0|
| 0  0  0  0|
 ‾‾‾‾‾‾‾‾‾‾‾ 


In [215]:
def get_board(board):
    return [
        (board >> 60) & 0xF, (board >> 56) & 0xF, (board >> 52) & 0xF, (board >> 48) & 0xF, # Row 0
        (board >> 44) & 0xF, (board >> 40) & 0xF, (board >> 36) & 0xF, (board >> 32) & 0xF, # Row 1
        (board >> 28) & 0xF, (board >> 24) & 0xF, (board >> 20) & 0xF, (board >> 16) & 0xF, # Row 2
        (board >> 12) & 0xF, (board >> 8) & 0xF,  (board >> 4) & 0xF,  board & 0xF        # Row 3
    ]
test = make_test_board()
int_to_board(test)
get_board(test)

 ___________________________ 
|     0    512    512   8192|
|     4    128    512     64|
|    16  16384   4096  16384|
| 32768      8  32768  16384|
 ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾ 


[0, 9, 9, 13, 2, 7, 9, 6, 4, 14, 12, 14, 15, 3, 15, 14]

In [344]:
def generate_next(board):
    empty_list = []
    for i in range(16):
        if (board >> (i * 4)) & 0xF == 0:
            empty_list.append(i)
    if not empty_list:
        return board
    
    pos = random.choice(empty_list)
    return board | 0b0001 << pos * 4

In [347]:
test = make_test_board()
int_to_board(test)

 ___________________________ 
| 32768      0   4096   2048|
|     8    128   4096   8192|
|  8192     64  16384      8|
|   128      0   4096     64|
 ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾ 


In [361]:
int_to_board(generate_next(test))

 ___________________________ 
| 32768      0   4096   2048|
|     8    128   4096   8192|
|  8192     64  16384      8|
|   128      2   4096     64|
 ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾ 
